In [1]:
import eval_beir_helpers
from redis_bm25 import gather_bm25_results
from redis_vector import gather_res_vector
from beir.retrieval.evaluation import EvaluateRetrieval

/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Eval beir datasets against Redis with different search techniques

### Load dataset

In [2]:
from redisvl.utils.vectorize import HFTextVectorizer

# set vectorizer
emb_model = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
# set dataset of interest
dataset = "nfcorpus"

corpus, queries, qrels = eval_beir_helpers.get_beir_dataset(dataset)
corpus_data = eval_beir_helpers.process_corpus(corpus, emb_model=emb_model)

./beir_datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

In [4]:
index = eval_beir_helpers.create_redis_index(dataset)

In [5]:
# load if need be
index.load(corpus_data)

['nfcorpus:01JQ9BWNS8MBHGRHCPK7A2JZE3',
 'nfcorpus:01JQ9BWNS8QAHXEMY27W09FMRE',
 'nfcorpus:01JQ9BWNS8MDTB1734T6YPP6Y7',
 'nfcorpus:01JQ9BWNS8YECRE88XME9N131X',
 'nfcorpus:01JQ9BWNS8XKSCZZS7QKCA25R4',
 'nfcorpus:01JQ9BWNS8KGC7EXEBYKCXN2J0',
 'nfcorpus:01JQ9BWNS81BMKANND2MJ5YCE2',
 'nfcorpus:01JQ9BWNS8K16K3Z9MBCDF1E2Z',
 'nfcorpus:01JQ9BWNS8W1D6F3CATC789Q1J',
 'nfcorpus:01JQ9BWNS8VE8EDE0R6BBPXHFD',
 'nfcorpus:01JQ9BWNS8HCN5BQ5MT9DR3YDV',
 'nfcorpus:01JQ9BWNS8JCZY6JRN84YEE46X',
 'nfcorpus:01JQ9BWNS8D260XR1XR6X7FV29',
 'nfcorpus:01JQ9BWNS80WC9ZJ9KD8EN1ECJ',
 'nfcorpus:01JQ9BWNS8W088GXQ831G44NDQ',
 'nfcorpus:01JQ9BWNS89DCV5S745GN49S3W',
 'nfcorpus:01JQ9BWNS8V56S2PQF7SKGAY6V',
 'nfcorpus:01JQ9BWNS8EHR2EDHVMRJZQVX7',
 'nfcorpus:01JQ9BWNS8PKBKEDGKH3F3YGBK',
 'nfcorpus:01JQ9BWNS8XJFZ9M3MDB6DFT7A',
 'nfcorpus:01JQ9BWNS8YTP3RMTB8EQP3TA0',
 'nfcorpus:01JQ9BWNS8TGZ370JC9QGKB650',
 'nfcorpus:01JQ9BWNS8342A2T991J4MDGN6',
 'nfcorpus:01JQ9BWNS89SJKT2FNFXYPDRRM',
 'nfcorpus:01JQ9BWNS8JAT5R15FV6359NNB',


### Check index info

See if all docs indexed, if there were any indexing errors, and view the total indexing time.

In [6]:
num_docs = index.info()["num_docs"]
total_indexing_time = round(float(index.info()["total_indexing_time"]) / 1000, 3) # indexing time in ms
indexing_failures = index.info()['Index Errors']

print(f"{num_docs=}, {total_indexing_time=} s, {indexing_failures=}")

num_docs=3633, total_indexing_time=3.398 s, indexing_failures=['indexing failures', 0, 'last indexing error', 'N/A', 'last indexing error key', 'N/A']


## Eval for different search methods

In [7]:
# simple bm25

redis_res_bm25 = gather_bm25_results(queries, index)
redis_bm25_ndcg, _map, redis_bm25_recall, redis_bm25_precision = (
    EvaluateRetrieval.evaluate(qrels, redis_res_bm25, [1, 10])
)

print(f"{redis_bm25_ndcg=} \n {redis_bm25_recall=} \n {redis_bm25_precision=}")

redis_bm25_ndcg={'NDCG@1': 0.38545, 'NDCG@10': 0.28788} 
 redis_bm25_recall={'Recall@1': 0.05404, 'Recall@10': 0.13888} 
 redis_bm25_precision={'P@1': 0.39938, 'P@10': 0.20464}


In [8]:
# basic vector

redis_res_vector = gather_res_vector(queries, index, emb_model)
vec_ndcg, _map, vec_recall, vec_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_vector, [1, 10]
)

print(f"{vec_ndcg=} \n {vec_recall=} \n {vec_precision=}")

vec_ndcg={'NDCG@1': 0.35294, 'NDCG@10': 0.27845} 
 vec_recall={'Recall@1': 0.02977, 'Recall@10': 0.12401} 
 vec_precision={'P@1': 0.37152, 'P@10': 0.22663}


In [9]:
# lin combo
from redis_lin_combo import gather_lin_combo_results

alpha = 0.7
redis_res_lin_combo = gather_lin_combo_results(queries, index, alpha, emb_model)
lin_combo_ndcg, _map, lin_combo_recall, lin_combo_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_lin_combo, [1, 10]
)

print(f"{lin_combo_ndcg=} \n {lin_combo_recall=} \n {lin_combo_precision=}")

lin_combo_ndcg={'NDCG@1': 0.4257, 'NDCG@10': 0.32395} 
 lin_combo_recall={'Recall@1': 0.05167, 'Recall@10': 0.15499} 
 lin_combo_precision={'P@1': 0.44582, 'P@10': 0.24334}


In [10]:
# weighted rrf
from redis_weighted_rrf import gather_weighted_rrf

redis_res_w_rrf = gather_weighted_rrf(queries, index, emb_model)
w_rrf_ndcg, _map, w_rrf_recall, w_rrf_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_w_rrf, [1, 10]
)

print(f"{w_rrf_ndcg=} \n {w_rrf_recall=} \n {w_rrf_precision=}")

w_rrf_ndcg={'NDCG@1': 0.41796, 'NDCG@10': 0.31793} 
 w_rrf_recall={'Recall@1': 0.04953, 'Recall@10': 0.16126} 
 w_rrf_precision={'P@1': 0.44272, 'P@10': 0.23622}


In [11]:
# weighted rrf
from redis_rerank import gather_rerank_results

redis_res_rerank = gather_rerank_results(queries, index, emb_model)
rerank_ndcg, _map, rerank_recall, rerank_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_rerank, [1, 10]
)

print(f"{rerank_ndcg=} \n {rerank_recall=} \n {rerank_precision=}")

rerank_ndcg={'NDCG@1': 0.36997, 'NDCG@10': 0.31588} 
 rerank_recall={'Recall@1': 0.03699, 'Recall@10': 0.1629} 
 rerank_precision={'P@1': 0.387, 'P@10': 0.24861}


# Save outputs of run

In [12]:
import pandas as pd

res_data = {
    "Model": ["redis_BM25", "redis_vector", "lin_combo", "weighted_rrf", "rerank"],
    "NDCG@1": [redis_bm25_ndcg['NDCG@1'], vec_ndcg['NDCG@1'], lin_combo_ndcg['NDCG@1'], w_rrf_ndcg['NDCG@1'], rerank_ndcg['NDCG@1']],
    "NDCG@10": [redis_bm25_ndcg['NDCG@10'], vec_ndcg['NDCG@10'], lin_combo_ndcg['NDCG@10'], w_rrf_ndcg['NDCG@10'], rerank_ndcg['NDCG@10']],
    "Recall@1": [redis_bm25_recall['Recall@1'], vec_recall['Recall@1'], lin_combo_recall['Recall@1'], w_rrf_recall['Recall@1'], rerank_recall['Recall@1']],
    "Recall@10": [redis_bm25_recall['Recall@10'], vec_recall['Recall@10'], lin_combo_recall['Recall@10'], w_rrf_recall['Recall@10'], rerank_recall['Recall@10']],
    "Precision@1": [redis_bm25_precision['P@1'], vec_precision['P@1'], lin_combo_precision['P@1'], w_rrf_precision['P@1'], rerank_precision['P@1']],
    "Precision@10": [redis_bm25_precision['P@10'], vec_precision['P@10'], lin_combo_precision['P@10'], w_rrf_precision['P@10'], rerank_precision['P@10']]
}

df = pd.DataFrame(res_data)

In [13]:
df.to_csv(f"{dataset}_results.csv", index=False)

## Cleanup

In [14]:
index.delete()